# Text emotion model (MELD)

> **Recommended:** use `backend/training/train_text.py` for real training. It adds macro-F1
> selection, focal loss, layer-wise learning rates, recent-context encoding, dev calibration,
> early stopping and CSV domain adaptation for mental-health corpora. The cells below are the
> original minimal demo and still run.
>
> `python backend/training/train_text.py --dataset meld --epochs 5 --batch-size 16`


# Text emotion model (MELD)
Fine-tunes RoBERTa on MELD utterances with the previous turns as dialogue context.
Saves the best checkpoint (by dev weighted-F1) to `experiments/text/best.pt`, which the API loads.

**Setup:** extract MELD so that `datasets/MELD.Raw/train/train_sent_emo.csv` (and dev/test equivalents) exist.

In [3]:
import sys, json, random
from pathlib import Path

# Find the project root (the folder that contains `backend/`), wherever the notebook is run from.
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "backend").is_dir())
sys.path.insert(0, str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT)

import numpy as np
import torch, torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, classification_report
from tqdm.auto import tqdm

from backend.models.text_model import TextEmotionModel, save_checkpoint, EMOTIONS, DEFAULT_CONFIG
from backend.services.meld_text_data import load_meld_text, MELDTextDataset, make_collate

Project root: /home/nikhil/System Design Project/multimodel-emotion-detection


In [4]:
ROOT = PROJECT_ROOT / "datasets" / "MELD.Raw"
OUT = PROJECT_ROOT / "experiments" / "text"; OUT.mkdir(parents=True, exist_ok=True)

CFG = dict(DEFAULT_CONFIG)          # model_name, proj_dim, context_turns, max_length, ...
EPOCHS, BATCH_SIZE, LR, WARMUP = 5, 16, 2e-5, 0.1
SEED = 42

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device, CFG)

cpu {'model_name': 'roberta-base', 'num_labels': 7, 'proj_dim': 256, 'dropout': 0.1, 'context_turns': 2, 'max_length': 128}


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CFG["model_name"])
collate = make_collate(tokenizer, CFG["max_length"], CFG["context_turns"])

data = {s: MELDTextDataset(load_meld_text(ROOT, s, CFG["context_turns"])) for s in ["train", "dev", "test"]}
print({k: len(v) for k, v in data.items()})

loaders = {
    "train": DataLoader(data["train"], BATCH_SIZE, shuffle=True, collate_fn=collate),
    "dev":   DataLoader(data["dev"], 64, collate_fn=collate),
    "test":  DataLoader(data["test"], 64, collate_fn=collate),
}

# Class weights: sqrt-damped inverse frequency (full inverse frequency tends to hurt weighted-F1).
counts = np.bincount(data["train"].labels, minlength=len(EMOTIONS))
print(dict(zip(EMOTIONS, counts)))
weights = torch.tensor(np.sqrt(counts.sum() / (len(EMOTIONS) * counts)), dtype=torch.float).to(device)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

FileNotFoundError: [Errno 2] No such file or directory: '/home/nikhil/System Design Project/multimodel-emotion-detection/datasets/MELD.Raw/train/train_sent_emo.csv'

In [ ]:
model = TextEmotionModel(CFG["model_name"], CFG["num_labels"], CFG["proj_dim"], CFG["dropout"]).to(device)
loss_fn = nn.CrossEntropyLoss(weight=weights)

# Lower LR for the pretrained encoder than for the new head.
optim = torch.optim.AdamW([
    {"params": model.encoder.parameters(), "lr": LR},
    {"params": list(model.proj.parameters()) + list(model.classifier.parameters()), "lr": LR * 5},
], weight_decay=0.01)
total_steps = EPOCHS * len(loaders["train"])
sched = get_linear_schedule_with_warmup(optim, int(WARMUP * total_steps), total_steps)
use_amp = device == "cuda"
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

@torch.no_grad()
def evaluate(loader):
    model.eval()
    preds, labels = [], []
    for b in loader:
        b = {k: v.to(device) for k, v in b.items()}
        out = model(b["input_ids"], b["attention_mask"])
        preds += out["logits"].argmax(-1).tolist()
        labels += b["labels"].tolist()
    return f1_score(labels, preds, average="weighted"), preds, labels

In [ ]:
best_f1, history = 0.0, []
for epoch in range(EPOCHS):
    model.train()
    running = 0.0
    for b in tqdm(loaders["train"], desc=f"epoch {epoch+1}/{EPOCHS}"):
        b = {k: v.to(device) for k, v in b.items()}
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            out = model(b["input_ids"], b["attention_mask"])
            loss = loss_fn(out["logits"].float(), b["labels"])
        optim.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optim)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optim); scaler.update(); sched.step()
        running += loss.item()

    dev_f1, _, _ = evaluate(loaders["dev"])
    history.append({"epoch": epoch + 1, "train_loss": running / len(loaders["train"]), "dev_weighted_f1": dev_f1})
    print(history[-1])
    if dev_f1 > best_f1:
        best_f1 = dev_f1
        save_checkpoint(model, CFG, OUT / "best.pt")
        print("  saved best checkpoint")

## Test evaluation (best checkpoint)

In [ ]:
ckpt = torch.load(OUT / "best.pt", map_location=device)
model.load_state_dict(ckpt["state_dict"])

test_f1, preds, labels = evaluate(loaders["test"])
print(f"Test weighted-F1: {test_f1:.4f}")
print(classification_report(labels, preds, target_names=EMOTIONS, digits=3, zero_division=0))

json.dump({"config": CFG, "history": history, "dev_best_f1": best_f1, "test_weighted_f1": test_f1},
          open(OUT / "metrics.json", "w"), indent=2)

## Sanity check: inference + explanation

In [ ]:
from backend.models.text_model import TextEmotionPredictor
from backend.explainability.text_explainer import TextExplainer

pred = TextEmotionPredictor(OUT / "best.pt")
ctx = ["Ross: Did you break my sandwich?"]
text = "Joey: I can't believe you would even ask me that!"
print(pred.predict(text, ctx))
print(TextExplainer(pred).explain(text, ctx))